<a href="https://colab.research.google.com/github/Alamgir-JUST/Skill-Morph/blob/main/Fl_based_Drone_IDS_for_IID_Scenario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import matplotlib
# Set font size and family for the entire figure
matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['font.family'] = 'serif'

In [27]:
##Dataset Link: https://onlineacademiccommunity.uvic.ca/isot/2024/12/05/drone-datasets/

In [2]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Access the file in Google Drive
df = pd.read_csv("/content/drive/MyDrive/Drone IDS Dataset Processed/updated_drone_ids_dataset.csv")

In [4]:
df.shape

(301555, 63)

In [5]:
df

,ts,Payload_Length,Var_Payload,Protocol Type,Duration,Entropy,Drone_port,Rate,Srate,Drate,...,Radius,Covariance,Variance,Weight,DS status,Fragments,Sequence number,flow_idle_time,flow_active_time,Label
0,0.009172,-1.348014,-0.966191,-0.328831,0.765486,0.002137,-1.037673,-0.372968,-0.098917,-0.006215,...,-0.873068,-0.983958,0.973405,-0.986035,-0.299227,-0.266974,-0.264555,-0.010915,-0.444785,2
1,0.009173,-1.353516,-0.967686,-0.322009,0.972344,0.004068,-1.037673,-0.391071,-0.109245,-0.006215,...,-0.922930,-0.991757,0.730427,0.986792,-0.299227,-0.266974,-0.264555,-0.519668,-0.432370,2
2,0.009174,-1.376699,-0.958604,-0.328831,0.765486,-0.046176,-1.037673,-0.361644,-0.109251,-0.006215,...,-0.974530,-0.997553,0.730427,-0.986035,-0.299227,-0.266974,-0.264555,-0.519668,-0.423013,2
3,0.009175,-1.414617,-0.953677,-0.322009,0.558629,-0.181737,-1.037673,-0.391160,-0.109241,-0.006077,...,-0.877347,-0.981531,0.730427,0.986792,-0.299227,-0.266974,-0.264555,-0.010915,-0.421956,2
4,0.009175,-1.343496,-0.970667,-0.328831,0.765486,0.002311,-1.037673,-0.380858,-0.109248,-0.006215,...,-0.894850,-0.986919,0.973405,-0.986035,-0.299227,-0.266974,-0.264555,-0.519668,-0.405997,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301550,-0.923558,-1.209701,-0.592404,3.025035,-1.789361,-1.200121,-1.353331,-0.321559,-0.108466,-0.003638,...,-0.113462,-0.553236,0.730427,-1.116040,2.830843,3.684909,3.992016,-0.519668,1.369613,5
301551,-0.924390,-1.608923,-0.989828,3.025035,-1.789361,-1.200121,-1.353331,-0.391285,-0.109078,-0.006215,...,-1.061645,-1.012872,0.973405,1.116797,-0.299227,-0.266974,0.017758,-0.519668,-0.232059,5
301552,-0.924457,-1.454107,-0.851728,3.025035,-1.789361,-1.200121,-1.353331,-0.339860,-0.109087,0.004649,...,-0.966796,-0.991320,0.730427,-1.116040,2.439584,1.050320,-0.198301,-0.519668,2.254120,5
301553,-0.922854,-1.572773,-0.984650,3.025035,-1.789361,-1.200121,-1.353331,-0.362409,-0.108988,0.009998,...,-0.159449,-0.634068,0.973405,1.116797,2.048325,-0.266974,1.404848,-0.519668,1.244857,5


In [ ]:
df['Label'].value_counts()

,count
Label,
0,89648
9,77226
1,71898
5,23122
3,9732
4,8218
6,6099
7,5859
8,5011


In [6]:
# Print the names of the features (columns)
print(df.columns)

Index(['ts', 'Payload_Length', 'Var_Payload', 'Protocol Type', 'Duration',
       'Entropy', 'Drone_port', 'Rate', 'Srate', 'Drate', 'fin_flag_number',
       'syn_flag_number', 'rst_flag_number', 'psh_flag_number',
       'ack_flag_number', 'urg_flag_number', 'ece_flag_number',
       'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'urg_count',
       'rst_count', 'max_duration', 'min_duration', 'sum_duration',
       'average_duration', 'std_duration', 'CoAP', 'HTTP', 'HTTPS', 'DNS',
       'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP',
       'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size',
       'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance',
       'Weight', 'DS status', 'Fragments', 'Sequence number', 'flow_idle_time',
       'flow_active_time', 'Label'],
      dtype='object')


In [7]:
df['Binary_Label'] = df['Label'].apply(lambda x: 0 if x == 0 else 1)

In [8]:
df['Binary_Label'].value_counts()

,count
Binary_Label,
1,211907
0,89648


In [9]:
df = df.drop(columns=['Label'])

# **IID Scenario**

## **Model with Federated Averaging (FedAvg)**

In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## Data Preparation
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split the dataframe into 3 clients
client_1 = df_shuffled.iloc[:len(df_shuffled)//3]
client_2 = df_shuffled.iloc[len(df_shuffled)//3:2*len(df_shuffled)//3]
client_3 = df_shuffled.iloc[2*len(df_shuffled)//3:]

# Define the model
def create_model():
    model = models.Sequential([
        layers.Input(shape=(client_1.shape[1] - 1,)),  # Exclude 'Binary_Label'
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Federated Averaging function
def train_model_on_client(client_data, model, epochs=1, batch_size=32):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
    return model.get_weights()

# Federated Averaging procedure
def federated_averaging(client_data_list, model, epochs=1, batch_size=32):
    # Initialize weights from the first client
    global_weights = train_model_on_client(client_data_list[0], model, epochs, batch_size)

    # Federated averaging
    for client_data in client_data_list[1:]:
        client_weights = train_model_on_client(client_data, model, epochs, batch_size)

        # Average the weights from each client
        for i in range(len(global_weights)):
            global_weights[i] = (global_weights[i] + client_weights[i]) / 2

    # Set the global model weights
    model.set_weights(global_weights)
    return model

# Evaluate the model on a client's data and calculate multiple metrics
def evaluate_model_on_client(client_data, model):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Predict on the client dataset
    y_pred = (model.predict(X) > 0.5).astype("int32")

    # Calculate evaluation metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    return accuracy, precision, recall, f1, cm

# Initialize the model
model = create_model()

# Split data into clients
client_data_list = [client_1, client_2, client_3]

# Perform federated averaging
global_model = federated_averaging(client_data_list, model, epochs=1, batch_size=32)

# Evaluate on each client
accuracy_client_1, precision_client_1, recall_client_1, f1_client_1, cm_client_1 = evaluate_model_on_client(client_1, global_model)
accuracy_client_2, precision_client_2, recall_client_2, f1_client_2, cm_client_2 = evaluate_model_on_client(client_2, global_model)
accuracy_client_3, precision_client_3, recall_client_3, f1_client_3, cm_client_3 = evaluate_model_on_client(client_3, global_model)

# Print client evaluation metrics
print("Client 1 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_1}, Precision: {precision_client_1}, Recall: {recall_client_1}, F1-Score: {f1_client_1}")
print("Confusion Matrix:\n", cm_client_1)

print("\nClient 2 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_2}, Precision: {precision_client_2}, Recall: {recall_client_2}, F1-Score: {f1_client_2}")
print("Confusion Matrix:\n", cm_client_2)

print("\nClient 3 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_3}, Precision: {precision_client_3}, Recall: {recall_client_3}, F1-Score: {f1_client_3}")
print("Confusion Matrix:\n", cm_client_3)

# Evaluate global model on the entire dataset
accuracy_global, precision_global, recall_global, f1_global, cm_global = evaluate_model_on_client(df_shuffled, global_model)

# Print global model evaluation metrics
print("\nGlobal Model Evaluation Metrics:")
print(f"Accuracy: {accuracy_global}, Precision: {precision_global}, Recall: {recall_global}, F1-Score: {f1_global}")
print("Confusion Matrix:\n", cm_global)


3142/3142 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
Client 1 Evaluation Metrics:
Accuracy: 0.9998408245289401, Precision: 0.9998304557913475, Recall: 0.9999434788752296, F1-Score: 0.9998869641393732
Confusion Matrix:
 [[29736    12]
 [    4 70766]]

Client 2 Evaluation Metrics:
Accuracy: 0.99992041226447, Precision: 0.9999008975847326, Recall: 0.9999858413093957, F1-Score: 0.9999433676431029
Confusion Matrix:
 [[29883     7]
 [    1 70627]]

Client 3 Evaluation Metrics:
Accuracy: 0.9999005163202976, Precision: 0.9998723712012706, Recall: 0.9999858174133799, F1-Score: 0.9999290910895863
Confusion Matrix:
 [[30001     9]
 [    1 70508]]
9424/9424 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step

Global Model Evaluation Metrics:
Accuracy: 0.9998872510818921, Precision: 0.99986788028066, Recall: 0.9999716856923084, F1-Score: 0.9999197802923773
Confusion Matrix:
 [[ 89620     28]
 [     6 211901]]


## **Model with FedProx (Federated Proximal)**

In [26]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Data Preparation
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split the dataframe into 3 clients
client_1 = df_shuffled.iloc[:len(df_shuffled)//3]
client_2 = df_shuffled.iloc[len(df_shuffled)//3:2*len(df_shuffled)//3]
client_3 = df_shuffled.iloc[2*len(df_shuffled)//3:]

# Define the model
def create_model():
    model = models.Sequential([
        layers.Input(shape=(client_1.shape[1] - 1,)),  # Exclude 'Binary_Label'
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Federated Proximal function (FedProx)
def train_model_on_client(client_data, model, global_weights, epochs=1, batch_size=32, mu=0.1):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Get current model weights
    model.set_weights(global_weights)

    # Local training with FedProx adjustment
    local_model = model
    initial_weights = model.get_weights()

    # Compute gradients for the local model and apply FedProx regularization
    for epoch in range(epochs):
        local_model.fit(X, y, epochs=1, batch_size=batch_size, verbose=0)

        # Calculate FedProx regularization term (proximal term)
        local_weights = local_model.get_weights()

        # Apply FedProx term: (w_i - w) + mu * (w_i - w_previous)
        for i in range(len(local_weights)):
            local_weights[i] = local_weights[i] - initial_weights[i] + mu * (local_weights[i] - global_weights[i])

        # Update model weights after training and FedProx adjustment
        local_model.set_weights(local_weights)

    return local_model.get_weights()

# Federated Averaging procedure with FedProx
def federated_averaging_with_fedprox(client_data_list, model, epochs=1, batch_size=32, mu=0.1):
    # Initialize weights from the first client
    global_weights = train_model_on_client(client_data_list[0], model, model.get_weights(), epochs, batch_size, mu)

    # Federated averaging with FedProx adjustment
    for client_data in client_data_list[1:]:
        client_weights = train_model_on_client(client_data, model, global_weights, epochs, batch_size, mu)

        # Average the weights from each client
        for i in range(len(global_weights)):
            global_weights[i] = (global_weights[i] + client_weights[i]) / 2

    # Set the global model weights
    model.set_weights(global_weights)
    return model

# Evaluate the model on a client's data and calculate multiple metrics
def evaluate_model_on_client(client_data, model):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Predict on the client dataset
    y_pred = (model.predict(X) > 0.5).astype("int32")

    # Calculate evaluation metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    return accuracy, precision, recall, f1, cm

# Initialize the model
model = create_model()

# Split data into clients
client_data_list = [client_1, client_2, client_3]

# Perform federated averaging with FedProx
mu = 0.1  # FedProx hyperparameter (proximal regularization term)
global_model = federated_averaging_with_fedprox(client_data_list, model, epochs=1, batch_size=32, mu=mu)

# Evaluate on each client
accuracy_client_1, precision_client_1, recall_client_1, f1_client_1, cm_client_1 = evaluate_model_on_client(client_1, global_model)
accuracy_client_2, precision_client_2, recall_client_2, f1_client_2, cm_client_2 = evaluate_model_on_client(client_2, global_model)
accuracy_client_3, precision_client_3, recall_client_3, f1_client_3, cm_client_3 = evaluate_model_on_client(client_3, global_model)

# Print client evaluation metrics
print("Client 1 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_1}, Precision: {precision_client_1}, Recall: {recall_client_1}, F1-Score: {f1_client_1}")
print("Confusion Matrix:\n", cm_client_1)

print("\nClient 2 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_2}, Precision: {precision_client_2}, Recall: {recall_client_2}, F1-Score: {f1_client_2}")
print("Confusion Matrix:\n", cm_client_2)

print("\nClient 3 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_3}, Precision: {precision_client_3}, Recall: {recall_client_3}, F1-Score: {f1_client_3}")
print("Confusion Matrix:\n", cm_client_3)

# Evaluate global model on the entire dataset
accuracy_global, precision_global, recall_global, f1_global, cm_global = evaluate_model_on_client(df_shuffled, global_model)

# Print global model evaluation metrics
print("\nGlobal Model Evaluation Metrics:")
print(f"Accuracy: {accuracy_global}, Precision: {precision_global}, Recall: {recall_global}, F1-Score: {f1_global}")
print("Confusion Matrix:\n", cm_global)


3142/3142 ━━━━━━━━━━━━━━━━━━━━ 3s 953us/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Client 1 Evaluation Metrics:
Accuracy: 0.9996617521239977, Precision: 0.9995903607650366, Recall: 0.999929348594037, F1-Score: 0.9997598259444491
Confusion Matrix:
 [[29719    29]
 [    5 70765]]

Client 2 Evaluation Metrics:
Accuracy: 0.9997711852603514, Precision: 0.9997027474627372, Recall: 0.9999716826187914, F1-Score: 0.9998371969562909
Confusion Matrix:
 [[29869    21]
 [    2 70626]]

Client 3 Evaluation Metrics:
Accuracy: 0.9998408261124763, Precision: 0.9997873034329225, Recall: 0.9999858174133799, F1-Score: 0.9998865505700834
Confusion Matrix:
 [[29995    15]
 [    1 70508]]
9424/9424 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step

Global Model Evaluation Metrics:
Accuracy: 0.9997579214405332, Precision: 0.9996933441527807, Recall: 0.9999622475897446, F1-Score: 0.9998277777908845
Confusion Matrix:
 [[ 89583     65]
 [     8 211899]]


## **Model with FedNova (Federated Normalized Averaging)**

In [28]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Data Preparation
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split the dataframe into 3 clients
client_1 = df_shuffled.iloc[:len(df_shuffled)//3]
client_2 = df_shuffled.iloc[len(df_shuffled)//3:2*len(df_shuffled)//3]
client_3 = df_shuffled.iloc[2*len(df_shuffled)//3:]

# Define the model
def create_model():
    model = models.Sequential([
        layers.Input(shape=(client_1.shape[1] - 1,)),  # Exclude 'Binary_Label'
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Federated Normalized Averaging (FedNova) function
def train_model_on_client(client_data, model, global_weights, epochs=1, batch_size=32):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Get current model weights
    model.set_weights(global_weights)

    # Local training
    initial_weights = model.get_weights()
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
    local_weights = model.get_weights()

    # Calculate normalization term (norm of the gradients or updates)
    delta_weights = [local_weights[i] - initial_weights[i] for i in range(len(local_weights))]
    norm_factor = np.sqrt(sum(np.linalg.norm(delta_weights[i]) ** 2 for i in range(len(delta_weights))))

    # Normalize the local updates
    if norm_factor > 0:
        delta_weights = [delta_weights[i] / norm_factor for i in range(len(delta_weights))]

    return delta_weights

# Federated Normalized Averaging procedure
def federated_averaging_with_fednova(client_data_list, model, epochs=1, batch_size=32):
    # Initialize global weights
    global_weights = train_model_on_client(client_data_list[0], model, model.get_weights(), epochs, batch_size)

    # FedNova - Normalized Averaging
    for client_data in client_data_list[1:]:
        client_weights = train_model_on_client(client_data, model, global_weights, epochs, batch_size)

        # Average the normalized weights from each client
        for i in range(len(global_weights)):
            global_weights[i] += client_weights[i]

    # Normalize the global model weights (sum of normalized local updates)
    global_model_norm = np.sqrt(sum(np.linalg.norm(global_weights[i]) ** 2 for i in range(len(global_weights))))
    if global_model_norm > 0:
        global_weights = [global_weights[i] / global_model_norm for i in range(len(global_weights))]

    # Set the global model weights
    model.set_weights(global_weights)
    return model

# Evaluate the model on a client's data and calculate multiple metrics
def evaluate_model_on_client(client_data, model):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Predict on the client dataset
    y_pred = (model.predict(X) > 0.5).astype("int32")

    # Calculate evaluation metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    return accuracy, precision, recall, f1, cm

# Initialize the model
model = create_model()

# Split data into clients
client_data_list = [client_1, client_2, client_3]

# Perform federated averaging with FedNova
global_model = federated_averaging_with_fednova(client_data_list, model, epochs=1, batch_size=32)

# Evaluate on each client
accuracy_client_1, precision_client_1, recall_client_1, f1_client_1, cm_client_1 = evaluate_model_on_client(client_1, global_model)
accuracy_client_2, precision_client_2, recall_client_2, f1_client_2, cm_client_2 = evaluate_model_on_client(client_2, global_model)
accuracy_client_3, precision_client_3, recall_client_3, f1_client_3, cm_client_3 = evaluate_model_on_client(client_3, global_model)

# Print client evaluation metrics
print("Client 1 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_1}, Precision: {precision_client_1}, Recall: {recall_client_1}, F1-Score: {f1_client_1}")
print("Confusion Matrix:\n", cm_client_1)

print("\nClient 2 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_2}, Precision: {precision_client_2}, Recall: {recall_client_2}, F1-Score: {f1_client_2}")
print("Confusion Matrix:\n", cm_client_2)

print("\nClient 3 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_3}, Precision: {precision_client_3}, Recall: {recall_client_3}, F1-Score: {f1_client_3}")
print("Confusion Matrix:\n", cm_client_3)

# Evaluate global model on the entire dataset
accuracy_global, precision_global, recall_global, f1_global, cm_global = evaluate_model_on_client(df_shuffled, global_model)

# Print global model evaluation metrics
print("\nGlobal Model Evaluation Metrics:")
print(f"Accuracy: {accuracy_global}, Precision: {precision_global}, Recall: {recall_global}, F1-Score: {f1_global}")
print("Confusion Matrix:\n", cm_global)


3142/3142 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 3s 959us/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
Client 1 Evaluation Metrics:
Accuracy: 0.957092262082413, Precision: 0.9425569036932462, Recall: 1.0, F1-Score: 0.9704291307000884
Confusion Matrix:
 [[25435  4313]
 [    0 70770]]

Client 2 Evaluation Metrics:
Accuracy: 0.9562864362601723, Precision: 0.9414422820581179, Recall: 0.9999858413093957, F1-Score: 0.969831374272218
Confusion Matrix:
 [[25497  4393]
 [    1 70627]]

Client 3 Evaluation Metrics:
Accuracy: 0.9559088331559208, Precision: 0.9408601433127394, Recall: 1.0, F1-Score: 0.9695290477827432
Confusion Matrix:
 [[25578  4432]
 [    0 70509]]
9424/9424 ━━━━━━━━━━━━━━━━━━━━ 9s 968us/step

Global Model Evaluation Metrics:
Accuracy: 0.9564291754406328, Precision: 0.9416203053625068, Recall: 0.999995280948718, F1-Score: 0.9699302667804857
Confusion Matrix:
 [[ 76510  13138]
 [     1 211906]]


## **Model with FedDyn (Federated Dynamics)**

In [29]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Data Preparation
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split the dataframe into 3 clients
client_1 = df_shuffled.iloc[:len(df_shuffled)//3]
client_2 = df_shuffled.iloc[len(df_shuffled)//3:2*len(df_shuffled)//3]
client_3 = df_shuffled.iloc[2*len(df_shuffled)//3:]

# Define the model
def create_model():
    model = models.Sequential([
        layers.Input(shape=(client_1.shape[1] - 1,)),  # Exclude 'Binary_Label'
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Federated Dynamics (FedDyn) function
def train_model_on_client(client_data, model, global_weights, epochs=1, batch_size=32):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Get current model weights
    model.set_weights(global_weights)

    # Local training
    initial_weights = model.get_weights()
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
    local_weights = model.get_weights()

    # Compute the gradient dynamics (change in gradients or weights)
    weight_diff = [local_weights[i] - initial_weights[i] for i in range(len(local_weights))]

    return weight_diff

# Federated Dynamics (FedDyn) procedure
def federated_dynamics(client_data_list, model, epochs=1, batch_size=32):
    # Initialize global weights from the first client
    global_weights = train_model_on_client(client_data_list[0], model, model.get_weights(), epochs, batch_size)

    # FedDyn - Dynamic Aggregation
    for client_data in client_data_list[1:]:
        # Get the weight differences from each client
        client_weight_diff = train_model_on_client(client_data, model, global_weights, epochs, batch_size)

        # Dynamically scale updates based on the weight differences
        for i in range(len(global_weights)):
            # Here we can introduce dynamic scaling based on the change magnitude
            weight_magnitude = np.linalg.norm(client_weight_diff[i])
            if weight_magnitude > 0:
                client_weight_diff[i] = client_weight_diff[i] / weight_magnitude
            # Aggregate the weighted updates from each client
            global_weights[i] += client_weight_diff[i]

    # Normalize the aggregated weights (global model)
    global_weights_norm = np.sqrt(sum(np.linalg.norm(global_weights[i]) ** 2 for i in range(len(global_weights))))
    if global_weights_norm > 0:
        global_weights = [global_weights[i] / global_weights_norm for i in range(len(global_weights))]

    # Set the global model weights
    model.set_weights(global_weights)
    return model

# Evaluate the model on a client's data and calculate multiple metrics
def evaluate_model_on_client(client_data, model):
    X = client_data.drop(columns=['Binary_Label'])
    y = client_data['Binary_Label']

    # Predict on the client dataset
    y_pred = (model.predict(X) > 0.5).astype("int32")

    # Calculate evaluation metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    return accuracy, precision, recall, f1, cm

# Initialize the model
model = create_model()

# Split data into clients
client_data_list = [client_1, client_2, client_3]

# Perform federated dynamics (FedDyn)
global_model = federated_dynamics(client_data_list, model, epochs=1, batch_size=32)

# Evaluate on each client
accuracy_client_1, precision_client_1, recall_client_1, f1_client_1, cm_client_1 = evaluate_model_on_client(client_1, global_model)
accuracy_client_2, precision_client_2, recall_client_2, f1_client_2, cm_client_2 = evaluate_model_on_client(client_2, global_model)
accuracy_client_3, precision_client_3, recall_client_3, f1_client_3, cm_client_3 = evaluate_model_on_client(client_3, global_model)

# Print client evaluation metrics
print("Client 1 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_1}, Precision: {precision_client_1}, Recall: {recall_client_1}, F1-Score: {f1_client_1}")
print("Confusion Matrix:\n", cm_client_1)

print("\nClient 2 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_2}, Precision: {precision_client_2}, Recall: {recall_client_2}, F1-Score: {f1_client_2}")
print("Confusion Matrix:\n", cm_client_2)

print("\nClient 3 Evaluation Metrics:")
print(f"Accuracy: {accuracy_client_3}, Precision: {precision_client_3}, Recall: {recall_client_3}, F1-Score: {f1_client_3}")
print("Confusion Matrix:\n", cm_client_3)

# Evaluate global model on the entire dataset
accuracy_global, precision_global, recall_global, f1_global, cm_global = evaluate_model_on_client(df_shuffled, global_model)

# Print global model evaluation metrics
print("\nGlobal Model Evaluation Metrics:")
print(f"Accuracy: {accuracy_global}, Precision: {precision_global}, Recall: {recall_global}, F1-Score: {f1_global}")
print("Confusion Matrix:\n", cm_global)


3142/3142 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step  
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
3142/3142 ━━━━━━━━━━━━━━━━━━━━ 3s 944us/step
Client 1 Evaluation Metrics:
Accuracy: 0.9996319067231739, Precision: 0.9998304054722501, Recall: 0.9996467429701851, F1-Score: 0.9997385657860337
Confusion Matrix:
 [[29736    12]
 [   25 70745]]

Client 2 Evaluation Metrics:
Accuracy: 0.9997114944587039, Precision: 0.9999291874973445, Recall: 0.999660191425497, F1-Score: 0.9997946713679843
Confusion Matrix:
 [[29885     5]
 [   24 70604]]

Client 3 Evaluation Metrics:
Accuracy: 0.9997114973288632, Precision: 0.9999716247659043, Recall: 0.999617070161256, F1-Score: 0.9997943160298738
Confusion Matrix:
 [[30008     2]
 [   27 70482]]
9424/9424 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step

Global Model Evaluation Metrics:
Accuracy: 0.9996849662582282, Precision: 0.9999103139013453, Recall: 0.9996413521025733, F1-Score: 0.999775814912792
Confusion Matrix:
 [[ 89629     19]
 [    76 211831]]


# **Thank You**